# Tutorial Assignment: Version Control, Collaboration, Git LFS, DVC, and Docker for MLOps

## Learning Objective
This tutorial assignment gives you hands-on practice with the core engineering tools required for reproducible MLOps workflows. You will create a small ML-style project, version the code using Git, isolate changes using branches, resolve a merge conflict, handle large model artifacts using Git LFS, version datasets using DVC, and containerize a small prediction service using Docker.

## Context
In normal software projects, code is usually the main artifact. In machine learning projects, the final result depends on five moving parts: code, data, configuration, environment, and randomness. A model can change even when the code is unchanged if the dataset, hyperparameters, dependency versions, or random seeds change. Therefore, an MLOps workflow must track not only source code, but also data, models, configs, and the execution environment.

By the end of this notebook, you should be able to explain and implement a minimal reproducible ML workflow using Git, GitHub-style collaboration, Git LFS, DVC, and Docker.

## Submission Instructions
Complete every task cell marked `TODO`. The solutions are provided at the bottom of this notebook for self-checking after completion.

## Setup Cell
Run this cell only if you are working in a fresh Python environment. Shell commands are written with `!` because this notebook is intended to be executed from Jupyter/Colab-style environments.

If Docker, Git LFS, or DVC are unavailable in your environment, still write the expected commands and explain what they would do.

In [ ]:
import os
import json
import textwrap
from pathlib import Path

PROJECT_DIR = Path("mlops_versioning_demo")
PROJECT_DIR.mkdir(exist_ok=True)
os.chdir(PROJECT_DIR)
print("Working directory:", Path.cwd())

# Part 1 — Create a Minimal ML Project

## Task 1: Build the Initial Project Structure

### Problem Description
You are starting a small ML project that predicts customer churn from a tiny CSV dataset. Before training any model, you need a clean project layout that separates source code, data, model outputs, and configuration. This separation is important because Git, DVC, Git LFS, and Docker will later track different parts of the project differently.

Create the following structure:

```text
mlops_versioning_demo/
├── src/
│   ├── prepare.py
│   └── train.py
├── data/
│   └── dataset.csv
├── models/
├── params.json
├── README.md
└── .gitignore
```

### Student Task
Write Python code to create the folders and files. The dataset should contain at least five rows with columns:

```text
customer_id,tenure_months,monthly_charge,contract,churn
```

The `.gitignore` should ignore Python cache files, virtual environments, raw CSV files, and model binaries.

# Part 2 — Git Basics: Snapshot, Diff, Commit, and History

## Task 2: Initialize Git and Create the First Commit

### Problem Description
Git is a distributed version control system. It tracks changes in text-based files such as Python scripts, configs, and documentation. The core local workflow is:

```text
working directory → staging area → local repository
```

You edit files in the working directory, select what should be saved using `git add`, and then create a snapshot using `git commit`.

### Student Task
Initialize a Git repository, inspect the status, stage all project files, commit them, and print the one-line commit history.

## Task 3: Modify a Hyperparameter and Inspect the Difference

### Problem Description
In ML, hyperparameters are part of the experiment definition. Changing the learning rate changes the model training behavior, so the change must be traceable. Before committing, use `git diff` to inspect exactly what changed.

### Student Task
Change the learning rate in `params.json` from `0.01` to `0.05`, inspect the diff, commit the change, and print the commit history.

# Part 3 — Branching, Pull Requests, and Merge Conflicts

## Task 4: Create a Feature Branch for a README Update

### Problem Description
A branch is a movable pointer to a commit. Branches allow teams to isolate work. In GitHub Flow, developers usually create a short-lived branch, push it, open a pull request, review it, and merge it into `main`.

### Student Task
Create a branch called `fix/readme-description`, update the README to clarify that this is a hands-on MLOps demo, commit the change, switch back to `main`, merge the branch, and inspect the graph.

## Task 5: Create and Resolve a Merge Conflict

### Problem Description
A merge conflict happens when two branches edit the same line differently. Git cannot decide which version is correct, so the developer must manually resolve the conflict. In ML projects, conflicts frequently occur in config files such as `params.json` when two experiments tune the same hyperparameter.

### Student Task
Create two experiment branches from the same base:

- `experiment/lr-high`: set `learning_rate = 0.1`
- `experiment/lr-low`: set `learning_rate = 0.001`

Merge `experiment/lr-high` into `main`, then merge `experiment/lr-low`. Resolve the conflict by choosing `learning_rate = 0.05`, commit the resolved file, and inspect the graph.

# Part 4 — Git LFS for Large Model Files

## Task 6: Track Model Files with Git LFS

### Problem Description
Git stores every version of every committed file in history. Large binary files such as `.h5`, `.pt`, or `.pkl` model checkpoints can quickly bloat a repository. Git LFS solves this by storing a small text pointer in Git while storing the actual binary blob in an LFS store.

### Student Task
Configure Git LFS to track Keras model files (`*.h5`). Create a dummy model file in `models/model.h5`, add the `.gitattributes` file and model file, commit them, then verify which files are tracked by LFS.

# Part 5 — DVC for Dataset and Pipeline Versioning

## Task 7: Initialize DVC and Track the Dataset

### Problem Description
Git LFS is useful for large files, but it does not model data lineage or ML pipelines. DVC is designed for versioning datasets, models, and reproducible pipelines. Git tracks small `.dvc` metadata files, while DVC stores actual data in its cache or remote storage.

### Student Task
Initialize DVC, track `data/dataset.csv`, commit the generated metadata, configure a local DVC remote, and push the data to the remote.

## Task 8: Update the Dataset and Roll Back to Version 1

### Problem Description
Dataset versions change over time. A reproducible ML workflow must be able to recover older versions. With Git + DVC, Git checks out the metadata pointer and DVC checks out the corresponding data content.

### Student Task
Append two new rows to `data/dataset.csv`, run `dvc status`, re-add the dataset with DVC, commit the new `.dvc` metadata, push to DVC remote, and then roll back to the previous dataset version.

## Task 9: Define a DVC Pipeline

### Problem Description
A DVC pipeline describes the process that transforms raw data into model artifacts. It stores the dependency graph in `dvc.yaml`, while `dvc.lock` records exact hashes of dependencies and outputs from the last successful run. When `dvc repro` is executed, only stages affected by changed inputs are re-run.

### Student Task
Create a two-stage pipeline:

1. `prepare`: runs `python src/prepare.py`, depends on `src/prepare.py` and `data/dataset.csv`, produces `data/prepared.csv`
2. `train`: runs `python src/train.py`, depends on `src/train.py`, `data/prepared.csv`, and `params.json`, produces `models/model.h5`

Run `dvc repro` and commit `dvc.yaml` and `dvc.lock`.

# Part 6 — Docker for Reproducible Environments

## Task 10: Create a Minimal Prediction Service

### Problem Description
Containers provide lightweight isolation and portability. A Docker image contains application code, dependencies, and runtime instructions. A running container is an instance of an image. Docker helps ensure that the service runs consistently across machines.

You will create a tiny FastAPI app with two endpoints:

- `/health`: confirms that the service is running
- `/predict`: accepts a JSON payload containing texts and returns simple positive/negative labels

### Student Task
Create `app.py` and `requirements.txt` for a minimal prediction API.

## Task 11: Write a Dockerfile and Run the Service

### Problem Description
A Dockerfile defines the image build process as a sequence of layers. Each instruction that changes the filesystem creates a new layer. A good Dockerfile keeps the image small, copies only required files, installs dependencies, and defines a clear command.

### Student Task
Write a Dockerfile that:

1. Uses `python:3.10-slim`
2. Sets `/app` as the working directory
3. Installs dependencies from `requirements.txt`
4. Copies `app.py`
5. Exposes port `8000`
6. Runs the app using Uvicorn

Then build and run the image.

## Task 12: Docker Debugging and Cleanup

### Problem Description
Containers and images consume disk space. In real projects, unused images, stopped containers, and build cache can grow quickly. Docker provides commands for inspecting containers, checking image sizes, viewing image layers, and pruning unused resources.

### Student Task
Run commands to inspect the container, list image sizes, view image history, stop/remove the container, and remove unused Docker resources.

# Part 7 — Conceptual Questions

Answer these briefly in your own words.

1. Why is code versioning alone insufficient for ML reproducibility?
2. What is the difference between Git LFS and DVC?
3. Why should raw datasets and model binaries usually not be committed directly to Git?
4. Why is one hypothesis per branch useful in ML experiments?
5. What is the difference between a Docker image and a Docker container?
6. How do namespaces and cgroups help containers provide isolation?
7. Why does Docker Compose become useful when an application has multiple services?